In [ ]:
def calculate_h_index(citations):
    """
    h-index 계산 함수
    :param citations: 논문 인용 횟수 리스트
    :return: h-index
    """
    citations.sort(reverse=True)  # 논문을 내림차순으로 정렬

    h_index = 0
    for i, citation in enumerate(citations):
        if citation >= i + 1:
            h_index = i + 1
        else:
            break

    return h_index

def calculate_i10_index(citations):
    """
    i10-index 계산 함수
    :param citations: 논문 인용 횟수 리스트
    :return: i10-index
    """
    i10_index = sum(1 for citation in citations if citation >= 10)
    return i10_index

def calculate_g_index(citations):
    """
    g-index 계산 함수
    :param citations: 논문 인용 횟수 리스트
    :return: g-index
    """
    citations.sort(reverse=True)  # 논문을 내림차순으로 정렬

    g_index = 0
    citation_sum = 0
    for i, citation in enumerate(citations):
        citation_sum += citation
        if citation_sum >= (i + 1)**2:
            g_index = i + 1

    return g_index


In [ ]:
import pandas as pd
import datetime

# Object 타입에서 숫자형 체크용
def is_numeric(value):
    try:
        float(value)  # float으로 변환 가능한 경우 숫자
        return True
    except (ValueError, TypeError):
        return False

def recentness(dt, pYears):
    rct = 0
    lct = 0
    if len(pYears) == 0:
        rct = 0
        lct = 0
    else:
        year_avg = sum(pYears) / len(pYears)
        if year_avg >= int(dt) - 5:
            lct = 1
        else:
            lct = max(round((1 - (int(dt) - 3 - year_avg) * 0.1), 2), 0)
        rct += len(pYears)  # 연도 개수

    return rct, lct

pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_rows', None)

# authors.all 파일 위치 주소
authors_file_path = r"./gsc_data/authors.all"

try:
    with open(authors_file_path, 'r', encoding='cp1252') as file:
        lines = file.read().splitlines()
        data = [line.split('|') for line in lines]
        columns = ["Valid", "Name", "Affiliation", "Email Domain", "Citation Count", "ID"]
        df_a = pd.DataFrame(data, columns=columns)  # authors 의 a
        print("변환 완료")
except FileNotFoundError:
    print("파일을 찾을 수 없습니다.")
except UnicodeDecodeError:
    print("파일을 ANSI(Windows-1252) 인코딩으로 열 수 없습니다.")

df_a_copy = df_a.copy()  # df_a의 복사본 생성

def fill_missing_values(row):
    if pd.isna(row['ID']):
        if pd.isna(row['Citation Count']):
            if pd.isna(row['Email Domain']):  # 1의 경우
                row['ID'] = row['Affiliation']
            else:  # 2, 3, 4 의 경우
                row['ID'] = row['Email Domain']
                if is_numeric(row['Affiliation']):  # 4의 경우
                    row['Citation Count'] = row['Affiliation']
                    row['Affiliation'] = None
                    row['Email Domain'] = None
                elif '.' in str(row['Affiliation']):  # 3의 경우
                    row['Email Domain'] = row['Affiliation']
                    row['Affiliation'] = None
                    row['Citation Count'] = None
                else:  # 2의 경우
                    row['Email Domain'] = None
        else:  # 5, 6, 7 의 경우
            row['ID'] = row['Citation Count']
            if is_numeric(row['Email Domain']):  # 6, 7 의 경우
                row['Citation Count'] = row['Email Domain']
                if '.' in str(row['Affiliation']):  # 7 의 경우
                    row['Email Domain'] = row['Affiliation']
                    row['Affiliation'] = None
                else:  # 6의 경우
                    row['Email Domain'] = None
            else:  # 5의 경우
                row['Citation Count'] = None
    return row

df_a_copy = df_a_copy.apply(fill_missing_values, axis=1)
# print(df_a_copy)


In [ ]:

import math
import os

folder_path = r"./gsc_data/DATA/"
dt = 2013
years_dict = {}

# 데이터 저장을 위한 리스트
all_scoquality = []
all_rct = []
all_lct = []

# df_a_copy의 각 행에서 1번 열의 값을 추출
for index, row in df_a_copy.iterrows():
    name = row[5]  # 1번 열의 값
   
    # 파일 경로 생성
    file_path = os.path.join(folder_path, name + "_.dat")
   
    # 파일이 존재하는지 확인
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as file:
            # 파일의 각 줄을 읽어 리스트로 저장
            lines = file.readlines()
 
        # 각 줄을 나누어서 리스트로 저장
        line_lists = [line.strip().split('|') for line in lines]
       
        #빈 리스트 생성
        citations = [] # 인용수
        years = [] # 연도
        scoquality = 0 # 품질
        for line_list in line_lists:
            if len(line_list) > 0:
                if int(line_list[1]) >= 1993:
                    citations.append(int(line_list[0]))
                    years.append(int(line_list[1]))
                    if line_list[1] in years_dict:
                        years_dict[line_list[1]] += 1
                    else:
                        years_dict[line_list[1]] = 1
                    scoquality += math.log(int(line_list[0]) * math.exp((int(line_list[1]) - dt) / 10) + 1)
 
        sum_citations = sum(citations)
        all_scoquality.append(scoquality)
        rct, lct = recentness(dt, years)
        all_lct.append(lct)
        all_rct.append(rct)
        #각 index 계산 후 df_a_copy에 새로운 열로 추가
        df_a_copy.at[index, "sum_citation"] = sum_citations
        df_a_copy.at[index, "Paper Count"] = len(citations)
        df_a_copy.at[index, "h-index"] = calculate_h_index(citations)
        df_a_copy.at[index, "i10-index"] = calculate_i10_index(citations)
        df_a_copy.at[index, "g-index"] = calculate_g_index(citations)
        df_a_copy.at[index, "scoquality"] = scoquality
        df_a_copy.at[index, "lct"] = lct
        df_a_copy.at[index, "rct"] = rct
    else:
        print(f"파일을 찾을 수 없습니다: {file_path}")

# 최대값 계산
max_scoquality = max(all_scoquality)
max_rct = max(all_rct)
max_lct = max(all_lct)

# 표준화 및 점수 계산
for index, row in df_a_copy.iterrows():
    norm_scoquality = row['scoquality'] / max_scoquality
    norm_rct = row['rct'] / max_rct
    norm_lct = row['lct'] / max_lct
    score = (norm_scoquality * 0.33) + (norm_rct * 0.33) + (norm_lct * 0.34)
    df_a_copy.at[index, "norm_scoquality"] = norm_scoquality
    df_a_copy.at[index, "norm_rct"] = norm_rct
    df_a_copy.at[index, "norm_lct"] = norm_lct
    df_a_copy.at[index, "score"] = score

df_a_copy['ID'] = 'id_' + df_a_copy['ID']

print(max_scoquality)
print(max_rct)
print(max_lct)

print(df_a_copy)
df_a_copy.head(20)
df_a_copy.to_csv('output.csv', index=False)
df_a_copy.to_csv('output_index.csv', index=True)
